# Embeddings Semánticos y Similitud con Sentence-Transformers

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS24**
* **Nomenclatura Oficial:** Procesamiento de Lenguaje Natural
* **Nombre de Trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

## Objetivo
Dominar el uso de embeddings de oraciones (Sentence-Transformers) y cálculo de similitud coseno para implementar sistemas de búsqueda semántica y agrupamiento en español.

## Resultados de aprendizaje
Al final de este notebook vas a poder:
1. Explicar la diferencia entre embeddings a nivel de palabra (Word2Vec) y a nivel de oración completa (Sentence-BERT).
2. Generar representaciones semánticas densas utilizando la librería `sentence-transformers` en español.
3. Implementar un buscador semántico y un clasificador por umbral de similitud coseno.



## Terminología clave (Microglosario)

* **✦ Sentence-Transformers:** Modificación de la arquitectura BERT entrenada mediante redes siamesas que genera embeddings densos de alta calidad para frases u oraciones completas.
* **✦ Similitud Coseno:** Medida geométrica que calcula el coseno del ángulo entre dos vectores en un espacio multidimensional, indicando qué tan alineada está su dirección semántica (independiente del tamaño del vector).
* **✦ Búsqueda Semántica:** Técnica de recuperación de información que busca documentos interpretando la intención conceptual y el significado del texto, en lugar de realizar coincidencia exacta de palabras clave.



## 1. Instalación e importación

Instalamos `sentence-transformers` y `scikit-learn`, y después cargamos las herramientas necesarias para trabajar con embeddings y similitud coseno.


In [ ]:
# Instalación de dependencias
!pip install -q sentence-transformers scikit-learn

In [ ]:
# Importaciones necesarias
from sentence_transformers import SentenceTransformer, util  # Librería principal para embeddings de oraciones
import numpy as np  # Operaciones numéricas con arrays
from sklearn.metrics.pairwise import cosine_similarity  # Cálculo de similitud coseno
import warnings
warnings.filterwarnings('ignore')  # Suprime advertencias para claridad del notebook

---

## 2. Carga del modelo preentrenado

Usamos `paraphrase-multilingual-MiniLM-L12-v2`, un modelo multilingüe que produce embeddings de 384 dimensiones y funciona bien para similitud entre oraciones.


In [ ]:
# Cargamos el modelo preentrenado
# Esto descarga el modelo (≈420 MB) la primera vez
modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print(f"Modelo cargado: {modelo.get_sentence_embedding_dimension()} dimensiones")
print(f"Máxima longitud de secuencia: {modelo.max_seq_length} tokens")

---

## 3. Generación de embeddings para un corpus pequeño y controlado

Primero vamos a trabajar con un conjunto corto de oraciones agrupadas por tema. Eso nos permite mirar si el espacio vectorial aproxima una intuición razonable de cercanía semántica.


In [ ]:
# Dataset de oraciones en español rioplatense
oraciones = [
    # Grupo 1: Tecnología y programación
    "El desarrollo de software requiere conocimientos de algoritmos y estructuras de datos.",
    "Programar aplicaciones necesita entender lógica computacional y diseño de sistemas.",
    "La inteligencia artificial transforma la manera en que procesamos información.",
    
    # Grupo 2: Comida y gastronomía
    "El asado argentino es una tradición familiar que se disfruta los domingos.",
    "Las empanadas de carne son un plato típico de la gastronomía rioplatense.",
    "Me encanta tomar mate con amigos en la plaza.",
    
    # Grupo 3: Deportes
    "El fútbol es el deporte más popular en Argentina y genera mucha pasión.",
    "Messi es considerado uno de los mejores jugadores de la historia del fútbol.",
    
    # Grupo 4: Educación
    "La educación pública y gratuita es un derecho fundamental en nuestro país.",
    "Estudiar en la universidad requiere dedicación y esfuerzo constante.",
    
    # Grupo 5: Clima (para contrastar)
    "Hoy hace un frío terrible, no se puede salir sin campera.",
    "El verano porteño es caluroso y húmedo, casi insoportable.",
]

print(f"Total de oraciones: {len(oraciones)}")

In [ ]:
# Generamos embeddings para todas las oraciones
# El modelo codifica cada oración en un vector denso de 384 dimensiones
embeddings = modelo.encode(
    oraciones,
    convert_to_tensor=False,  # Devuelve numpy arrays (más simple para este ejemplo)
    show_progress_bar=True    # Muestra barra de progreso
)

print(f"\nForma de los embeddings: {embeddings.shape}")
print(f"Cada embedding tiene {embeddings.shape[1]} dimensiones")
print(f"\nPrimeras 10 dimensiones del primer embedding:")
print(embeddings[0][:10])

### Interpretación inicial

- Si el resultado tiene forma `(12, 384)`, eso significa que obtuvimos 12 vectores, uno por oración.
- Cada vector tiene 384 dimensiones.
- No vamos a leer esas dimensiones una por una; lo importante es cómo se comparan entre sí.
- La utilidad aparece cuando dos oraciones de significado parecido quedan relativamente cerca en el espacio.


---

## 4. Cálculo de similitud coseno

La **similitud coseno** mide el ángulo entre dos vectores:
- **1.0**: Vectores idénticos (misma dirección)
- **0.0**: Vectores ortogonales (no relacionados)
- **-1.0**: Vectores opuestos

### Fórmula matemática:

$$
\text{similitud}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}
$$

Donde:
- $A \cdot B$ es el producto punto de los vectores
- $\|A\|$ y $\|B\|$ son las normas (magnitudes) de los vectores

In [ ]:
# Calculamos la matriz de similitud completa (todas vs todas)
matriz_similitud = cosine_similarity(embeddings)

print(f"Forma de la matriz de similitud: {matriz_similitud.shape}")
print(f"\nMatriz de similitud (primeras 5 oraciones):")
print(matriz_similitud[:5, :5])
print(f"\nLa diagonal es 1.0 porque cada oración es idéntica a sí misma")

### Búsqueda de las oraciones más similares

Para cada oración del corpus vamos a recuperar las dos oraciones con mayor similitud, excluyendo la oración consigo misma.


In [ ]:
print("=" * 100)
print("ORACIONES MÁS SIMILARES SEMÁNTICAMENTE")
print("=" * 100)

for i in range(len(oraciones)):
    oracion = oraciones[i]
    scores = matriz_similitud[i]
    indices_ordenados = np.argsort(scores)[::-1]

    indices_similares = []
    for indice in indices_ordenados:
        if indice != i:
            indices_similares.append(indice)
        if len(indices_similares) == 2:
            break

    print(f"
ORACIÓN {i + 1}:")
    print(f"   {oracion}")
    print("
   Más similares:")

    for posicion in range(len(indices_similares)):
        indice_similar = indices_similares[posicion]
        score = scores[indice_similar]
        print(f"   {posicion + 1}. [Similitud: {score:.3f}] {oraciones[indice_similar]}")

    print("-" * 100)


### Qué conviene observar

- Si las oraciones de programación se recuperan entre sí.
- Si las oraciones de gastronomía también forman un pequeño grupo.
- Si los temas muy distintos quedan con scores más bajos.

La pregunta importante no es si el número exacto es alto o bajo, sino si el orden relativo parece consistente con el significado.


---

## 5. Búsqueda semántica con una consulta nueva

Ahora vamos a usar el mismo espacio vectorial como un buscador. La consulta también se transforma en embedding y después se compara con las oraciones almacenadas.


In [ ]:
def buscar_semanticamente(consulta, top_k=3):
    """
    Busca las `top_k` oraciones más cercanas semánticamente a una consulta.
    """
    embedding_consulta = modelo.encode([consulta], convert_to_tensor=False)
    similitudes = cosine_similarity(embedding_consulta, embeddings)[0]
    indices_ordenados = np.argsort(similitudes)[::-1]

    resultados = []
    for indice in indices_ordenados:
        oracion = oraciones[indice]
        score = similitudes[indice]
        resultados.append((indice, oracion, score))
        if len(resultados) == top_k:
            break

    return resultados


In [ ]:
consulta1 = "¿Cómo aprendo a crear aplicaciones web?"

print("BÚSQUEDA SEMÁNTICA")
print("=" * 80)
print(f"Consulta: '{consulta1}'
")

resultados = buscar_semanticamente(consulta1, top_k=3)

for i in range(len(resultados)):
    _, oracion, score = resultados[i]
    print(f"{i + 1}. [Score: {score:.4f}] {oracion}")
    print()


In [ ]:
consulta2 = "¿Qué comida típica hay en Buenos Aires?"

print("BÚSQUEDA SEMÁNTICA")
print("=" * 80)
print(f"Consulta: '{consulta2}'
")

resultados = buscar_semanticamente(consulta2, top_k=3)

for i in range(len(resultados)):
    _, oracion, score = resultados[i]
    print(f"{i + 1}. [Score: {score:.4f}] {oracion}")
    print()


In [ ]:
consulta3 = "Hablame sobre Lionel Messi"

print("BÚSQUEDA SEMÁNTICA")
print("=" * 80)
print(f"Consulta: '{consulta3}'
")

resultados = buscar_semanticamente(consulta3, top_k=3)

for i in range(len(resultados)):
    _, oracion, score = resultados[i]
    print(f"{i + 1}. [Score: {score:.4f}] {oracion}")
    print()


### Ventajas de la búsqueda semántica

1. No depende de coincidencias exactas de palabras.
2. Puede relacionar reformulaciones o sinónimos.
3. Se puede escalar con índices vectoriales como FAISS.
4. Reutiliza la misma representación para búsqueda, agrupamiento y recomendación.

### Comparación con búsqueda léxica

| Característica | TF-IDF | Embeddings semánticos |
|----------------|--------|----------------------|
| Coincidencia exacta | Necesaria | No necesaria |
| Captura sinónimos | No | Sí |
| Maneja paráfrasis | No | Sí |
| Dimensionalidad | Alta | Reducida |
| Interpretabilidad | Mayor | Menor |


---

## 6. Detección de paráfrasis

Ahora vamos a mirar un uso clásico de estos embeddings: decidir si dos oraciones expresan aproximadamente la misma idea.


In [ ]:
pares_parafrasis = [
    (
        "El perro corrió rápidamente por el parque.",
        "Un canino se desplazó velozmente a través del área verde."
    ),
    (
        "La inteligencia artificial está revolucionando la industria tecnológica.",
        "La IA transforma radicalmente el sector de la tecnología."
    ),
    (
        "El libro fue escrito por un autor argentino.",
        "Un escritor de Argentina redactó este texto."
    )
]

par_contrastante = (
    "El clima está muy frío hoy.",
    "La programación requiere práctica constante."
)

print("DETECCIÓN DE PARÁFRASIS")
print("=" * 100)

for i in range(len(pares_parafrasis)):
    oracion1 = pares_parafrasis[i][0]
    oracion2 = pares_parafrasis[i][1]

    emb1 = modelo.encode([oracion1])
    emb2 = modelo.encode([oracion2])
    similitud = cosine_similarity(emb1, emb2)[0][0]

    print(f"
Par {i + 1} (paráfrasis):")
    print(f"  A: {oracion1}")
    print(f"  B: {oracion2}")
    print(f"  Similitud: {similitud:.4f}")

emb1 = modelo.encode([par_contrastante[0]])
emb2 = modelo.encode([par_contrastante[1]])
similitud = cosine_similarity(emb1, emb2)[0][0]

print("
" + "=" * 100)
print("Par contrastante (no paráfrasis):")
print(f"  A: {par_contrastante[0]}")
print(f"  B: {par_contrastante[1]}")
print(f"  Similitud: {similitud:.4f}")


### Umbral de decisión

En un sistema real, no alcanza con mirar el score a ojo. Hace falta definir un umbral y validarlo con datos etiquetados del dominio.

```python
UMBRAL_PARAFRASIS = 0.75

if similitud >= UMBRAL_PARAFRASIS:
    print("Son paráfrasis")
else:
    print("No son paráfrasis")
```

La elección del umbral depende del costo de falsos positivos y falsos negativos en tu aplicación.


---

## 7. Comparación con promedio de Word2Vec

Una alternativa clásica consistía en promediar embeddings de palabras. Esa estrategia puede ser útil, pero suele perder orden y parte del contexto.


In [ ]:
oracion_a = "El gato persiguió al ratón."
oracion_b = "El ratón persiguió al gato."

emb_a = modelo.encode([oracion_a])
emb_b = modelo.encode([oracion_b])
similitud = cosine_similarity(emb_a, emb_b)[0][0]

print("PRUEBA DE SENSIBILIDAD AL ORDEN")
print("=" * 80)
print(f"Oración A: {oracion_a}")
print(f"Oración B: {oracion_b}")
print(f"
Similitud: {similitud:.4f}")
print("
Lectura: un promedio simple de Word2Vec tendería a acercarlas demasiado porque comparte casi las mismas palabras.")
print("Sentence-BERT conserva mejor la diferencia entre quién persigue a quién.")


---

## 8. Aplicaciones en el mundo real

### 1. Sistemas de preguntas frecuentes
Un usuario formula una pregunta y el sistema recupera la respuesta más cercana por significado.

### 2. Detección de plagio académico
Los embeddings permiten comparar párrafos aunque no repitan las mismas palabras exactas.

### 3. Recomendación de contenido
Se pueden sugerir textos o artículos cercanos a los intereses del usuario.

### 4. Agrupamiento de tickets de soporte
Consultas parecidas pueden agruparse aunque estén redactadas de formas distintas.

### 5. Búsqueda en documentos especializados
En ámbitos legales, médicos o técnicos, la cercanía semántica suele ser más útil que la coincidencia literal.


---

## Guía teórico-conceptual

### ¿Qué son los embeddings semánticos?

Son representaciones vectoriales densas de textos que capturan, de manera aproximada, relaciones de significado. No describen el texto con reglas manuales, sino con patrones aprendidos en grandes corpus.

### Evolución breve

#### 1. Era pre-neural
- Bag of Words y TF-IDF.
- Buen rendimiento para tareas léxicas.
- Dificultad para capturar similitud fina.

#### 2. Embeddings de palabras
- Word2Vec, GloVe y FastText.
- Una palabra recibe un vector.
- El problema aparece cuando queremos resumir oraciones completas.

#### 3. Embeddings contextuales
- BERT y variantes posteriores.
- El vector depende del contexto.
- Sentence-BERT ajusta esta idea para comparación eficiente de oraciones.

### ¿Cómo funciona Sentence-BERT?

1. Un transformer procesa la oración.
2. Se aplica una estrategia de `pooling` para obtener un solo vector.
3. Ese vector se compara con otros usando una métrica como similitud coseno.

La clave práctica es que una vez generado el embedding de cada documento, la búsqueda posterior es mucho más barata que volver a comparar pares completos con BERT clásico.


##  Preguntas y respuestas para estudio

### Conceptos fundamentales

**1. ¿Qué diferencia hay entre embeddings de palabras (Word2Vec) y embeddings de oraciones (Sentence-BERT)?**

Word2Vec genera un único vector fijo por palabra, sin considerar el contexto en que aparece. Para representar oraciones, se debe promediar los vectores de palabras, perdiendo orden y estructura sintáctica. Sentence-BERT, en cambio, usa transformers para generar un embedding único que captura el significado completo de la oración, considerando contexto, orden de palabras y relaciones sintácticas.

---

**2. ¿Por qué la similitud coseno es preferida sobre la distancia euclidiana para comparar embeddings?**

La similitud coseno mide el ángulo entre vectores, siendo insensible a su magnitud. Esto es importante porque dos textos pueden tener significado similar pero diferente "intensidad" (longitud de vector). En embeddings semánticos, nos interesa la dirección (significado) más que la magnitud. Además, muchos modelos normalizan embeddings, haciendo que coseno y distancia euclidiana sean equivalentes.

---

**3. ¿Qué es el "pooling" en Sentence-BERT y por qué es necesario?**

Pooling es la operación que reduce múltiples embeddings de tokens (uno por palabra) a un único embedding de oración. BERT genera un vector por cada token de entrada; pooling agrega estos vectores en uno solo. Las estrategias comunes son: MEAN pooling (promedio), CLS pooling (token especial [CLS]), y MAX pooling (máximo por dimensión). Es necesario porque necesitamos un único vector de dimensión fija para comparar oraciones de diferente longitud.

---

**4. ¿Cómo se entrena un modelo Sentence-BERT?**

Se usa una arquitectura Siamese (gemela): dos instancias del mismo BERT procesan pares de oraciones independientemente, generando dos embeddings. Una función de pérdida (triplet loss o contrastive loss) entrena el modelo para que oraciones similares (paráfrasis) tengan embeddings cercanos y oraciones diferentes tengan embeddings lejanos. Los datos de entrenamiento consisten en pares de oraciones con etiquetas de similitud.

---

**5. ¿Qué significa que un modelo sea "multilingüe"?**

Un modelo multilingüe fue entrenado con textos de múltiples idiomas simultáneamente, aprendiendo un espacio vectorial compartido donde oraciones con significado similar en diferentes idiomas tienen embeddings cercanos. Esto permite búsqueda cross-lingual (buscar en español, encontrar resultados en inglés) y funciona razonablemente en idiomas con pocos datos de entrenamiento (zero-shot transfer).

---

### Aspectos técnicos

**6. ¿Por qué BERT estándar no es eficiente para búsqueda semántica?**

BERT estándar requiere procesar pares de oraciones juntas (concatenadas) para obtener un score de similitud. Para buscar en n documentos, necesitaríamos n forward passes completos. Con 10,000 documentos, esto es 10,000 veces más lento que Sentence-BERT, que procesa cada documento una vez, guarda su embedding, y luego solo calcula similitud coseno (operación simple) contra la consulta.

---

**7. ¿Qué es FAISS y cuándo se usa?**

FAISS (Facebook AI Similarity Search) es una librería para búsqueda eficiente de vecinos más cercanos en espacios vectoriales de alta dimensionalidad. Se usa cuando tenemos millones de embeddings y necesitamos búsqueda en tiempo real (milisegundos). Usa técnicas como cuantización, índices invertidos y particionamiento del espacio para lograr búsqueda aproximada mucho más rápida que comparación exhaustiva, con mínima pérdida de precisión.

---

**8. ¿Cuál es la dimensionalidad típica de embeddings de Sentence-BERT y por qué ese rango?**

Entre 384 y 1024 dimensiones típicamente. Es un balance entre:
- **Expresividad**: Más dimensiones → mayor capacidad de capturar matices semánticos
- **Eficiencia**: Menos dimensiones → menor costo de almacenamiento y cálculo
- **Generalización**: Demasiadas dimensiones pueden causar overfitting

384 dimensiones (MiniLM) es suficiente para la mayoría de aplicaciones; 1024 (modelos large) para tareas muy exigentes.

---

**9. ¿Qué significa "convert_to_tensor=False" en el método encode()?**

Controla el tipo de dato devuelto:
- `False`: Devuelve numpy arrays (más simple para análisis en Python)
- `True`: Devuelve tensores de PyTorch (más eficiente para operaciones GPU)

Para análisis exploratorio y cálculos con scikit-learn, numpy es suficiente. Para pipelines de deep learning o procesamiento masivo en GPU, tensores son mejores.

---

**10. ¿Por qué promediar vectores de Word2Vec pierde información de orden?**

Porque la suma/promedio es una operación **conmutativa**: `vec(A) + vec(B) + vec(C) = vec(C) + vec(A) + vec(B)`. Por lo tanto, "Juan ama a María" y "María ama a Juan" tendrían exactamente el mismo embedding promedio, aunque signifiquen cosas diferentes. Word2Vec no tiene mecanismo para capturar que la posición de "Juan" y "María" relativa al verbo "ama" cambia el significado.

---

### Aplicaciones prácticas

**11. ¿Cómo se implementaría un sistema de FAQ automático con embeddings semánticos?**

```python
# Paso 1: Preprocesamiento (una vez)
preguntas_faq = ["¿Cómo reseteo mi contraseña?", ...]
respuestas_faq = ["Haz click en 'Olvidé mi contraseña'...", ...]
embeddings_faq = modelo.encode(preguntas_faq)

# Paso 2: Cuando llega consulta del usuario
consulta = "No puedo acceder a mi cuenta"
emb_consulta = modelo.encode([consulta])

# Paso 3: Búsqueda del FAQ más similar
similitudes = cosine_similarity(emb_consulta, embeddings_faq)[0]
idx_mejor = np.argmax(similitudes)

# Paso 4: Devolver respuesta si la similitud supera umbral
if similitudes[idx_mejor] > 0.7:
    return respuestas_faq[idx_mejor]
else:
    return "No encontré respuesta, te derivo a un agente humano"
```

---

**12. ¿Qué es detección de paráfrasis y para qué sirve?**

Es identificar si dos textos diferentes expresan el mismo significado. Se calcula la similitud coseno entre sus embeddings; si supera un umbral (ej. 0.75), se consideran paráfrasis. Aplicaciones:
- **Detección de plagio**: Identificar textos copiados aunque se hayan cambiado palabras
- **Deduplicación**: Eliminar noticias/tweets repetidos con diferente redacción
- **Evaluación de traducción**: Verificar si la traducción preserva el significado
- **Augmentación de datos**: Generar variaciones de textos de entrenamiento

---

**13. ¿Por qué la búsqueda semántica es mejor que búsqueda por keywords para un buscador de artículos científicos?**

Porque:
1. **Terminología variable**: Un mismo concepto se describe con diferentes términos técnicos
2. **Sinónimos disciplinares**: "machine learning" = "aprendizaje automático" = "aprendizaje de máquina"
3. **Abstracts vs consultas**: Los usuarios buscan con preguntas naturales, los papers usan lenguaje formal
4. **Conceptos relacionados**: Buscar "redes neuronales" puede beneficiarse de encontrar papers sobre "deep learning" aunque no contengan el término exacto

Búsqueda semántica captura relaciones conceptuales que keywords no pueden.

---

**14. ¿Cómo combinarías TF-IDF con Sentence-BERT en un sistema de producción?**

**Pipeline híbrido de dos etapas**:

1. **Etapa 1 - Filtrado rápido con TF-IDF**:
   - Búsqueda en toda la base de datos (millones de docs)
   - Retorna top 1000 candidatos en milisegundos
   - Elimina 99.9% de documentos claramente irrelevantes

2. **Etapa 2 - Re-ranking semántico con SBERT**:
   - Calcula embeddings solo para los 1000 candidatos
   - Re-ordena por similitud coseno
   - Retorna top 10 finales

**Ventajas**: Combina velocidad de TF-IDF con precisión semántica de SBERT. **Usado por**: Google Scholar, Semantic Scholar, etc.

---

**15. ¿Qué sesgos pueden aparecer en embeddings semánticos y cómo mitigarlos?**

**Sesgos comunes**:
- **Género**: "doctor" más cercano a "hombre", "enfermera" a "mujer"
- **Étnicos/raciales**: Asociaciones negativas con ciertos grupos
- **Socioeconómicos**: Asunciones sobre profesiones y clases sociales

**Origen**: Datos de entrenamiento (Wikipedia, web crawls) reflejan sesgos sociales existentes.

**Mitigación**:
1. **Auditorías**: Probar embeddings con listas de términos sensibles
2. **Debiasing**: Técnicas como Bolukbasi et al. (2016) para neutralizar dimensiones de sesgo
3. **Datos balanceados**: Entrenar con corpus curados que contrarresten sesgos
4. **Transparencia**: Documentar limitaciones conocidas del modelo

---

## Ejercicios prácticos

### Ejercicio 1: búsqueda semántica en un dominio específico
Armá un corpus corto de medicina, educación o derecho y probá consultas libres.

### Ejercicio 2: agrupamiento de noticias
Tomá titulares de varios temas y verificá si la similitud ayuda a separar grupos.

### Ejercicio 3: detección de plagio simple
Creá una paráfrasis, un contraste y un caso ambiguo. Después compará los scores.

### Ejercicio 4: comparación de modelos
Cargá dos modelos distintos de `sentence-transformers` y compará velocidad y calidad aparente.

### Ejercicio 5: recomendación de películas o artículos
Usá embeddings para recomendar sinopsis o textos parecidos a una consulta nueva.


## Conclusión

En este notebook trabajamos con una idea central del PLN moderno: representar oraciones completas como vectores reutilizables.

Eso nos permitió:

1. Comparar significado sin depender solo de palabras exactas.
2. Implementar una búsqueda semántica simple.
3. Detectar paráfrasis de forma aproximada.
4. Entender por qué estos modelos resultan útiles para recuperación y recomendación.

## Qué sigue

A partir de acá, estos embeddings pueden convertirse en una pieza de trabajo para sistemas más grandes: buscadores semánticos, RAG, agrupamiento de documentos o recomendadores.
